#  Lab Week 11 —  Hugging Face Pipelines and Fine-Tuning
Let's finish the task of **Text classification** using IMDb sentiment analysis.

The goal is not only to run code, but also to understand the workflow:

- choose a pretrained model,
- load and preprocess a dataset,
- fine-tune the model,
- save the model,
- reload it with `pipeline()`,
- test the final system

## 0. Setup
Run the following cell first. In Google Colab, use **Runtime → Change runtime type → GPU** when possible.


In [3]:
!pip install transformers datasets evaluate "accelerate>=1.1.0"

import transformers
import accelerate
import torch

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)

transformers: 5.12.1
accelerate: 1.14.0
torch: 2.12.0+cu130


# Text Classification with IMDb

Let's try to use 
1. `pipeline()` for simple inference, 
2. `Trainer` for fine-tuning, 
3. `AutoTokenizer` for tokenization, and 
4. `AutoModelForSequenceClassification` classes for loading pretrained models.

We will use **DistilBERT** because it is smaller and faster than BERT.

In [4]:
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, #convert text into model inputs.
    DataCollatorWithPadding, #dynamically pad inputs to the longest sequence in a batch.
    AutoModelForSequenceClassification, #load a pretrained model for text classification.
    TrainingArguments, #define training settings.
    Trainer,#train or fine-tune the model.
    pipeline,
)

checkpoint = "distilbert-base-uncased"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.12.0+cu130
CUDA available: True


## 1 Baseline: Basic sentiment pipeline before fine-tuning
The simplest Hugging Face workflow: `pipeline()` function hides many steps: loading a model, loading a tokenizer, tokenizing the input, running inference, and converting logits into labels and scores.

In [5]:
basic_classifier = pipeline("sentiment-analysis")

test_texts = [
    "The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.",
    "The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.",
    "What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.",
    "It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.",
    "I can see what the director was trying to do, and I respect the ambition, but the final result just does not work.",
    "There are a few weak scenes and some awkward jokes, but overall this is a fun and surprisingly touching movie.",
    ]
    
test_results = basic_classifier(test_texts)

for text, result in zip(test_texts, test_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)



[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.
Result: {'label': 'POSITIVE', 'score': 0.999855637550354}
--------------------------------------------------------------------------------
Text: The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.
Result: {'label': 'NEGATIVE', 'score': 0.9990723133087158}
--------------------------------------------------------------------------------
Text: What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.
Result: {'label': 'NEGATIVE', 'score': 0.9998123049736023}
--------------------------------------------------------------------------------
Text: It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.
Result: {'label': 'POSITIVE', 'score': 0.9993501305580139}
---

## 2 Load IMDb dataset

IMDb is a binary sentiment classification dataset:

- label `0` = negative,
- label `1` = positive.

In [6]:
dataset = load_dataset("imdb")
dataset.shape

train_ds = dataset["train"] 
test_ds = dataset["test"] 

print(train_ds[0])
print(test_ds[0])

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /home/user/.cache/huggingface/datasets/imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Sat Jun 20 15:10:53 2026).


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 3 Tokenization

The tokenizer converts text into numerical token IDs.

1. For Transformer models, common tokenizer outputs include:
- `input_ids`: integer IDs for tokens,
- `attention_mask`: tells the model which tokens are real and which are padding.

2. Some parameters:
- `truncation=True`: cut very long reviews, 
- `padding="max_length"`: make all examples the same length, 
- `max_length=256`: reduce memory use.
- `batched=True` : tell Hugging Face to tokenize many examples at the same time, which is faster than inputs["texts"]

In [7]:
# Load the tokenizer for the pretrained model
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [8]:
def tokenize_function(inputs):
    return tokenizer(
        inputs["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# Hugging Face Trainer expects the target column to be named "labels".
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

print(train_ds[0])
print(train_ds[0].keys())



{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 4 Load model and define metric
- `checkpoint = "distilbert-base-uncased"`
- `AutoModel`: the model only returns hidden states — contextual vector representations of the input tokens. It does not output POSITIVE or NEGATIVE labels.
- `AutoModelForSequenceClassification`: We choose this for loading a Transformer model with an additional classification head. 

In [9]:
# This is a 2-class classification model.
# Class 0 means NEGATIVE. Class 1 means POSITIVE.
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

#pip install scikit-learn
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Fine-tune with `Trainer`

- `TrainingArguments` defines the training settings.
- `Trainer` runs the training loop for us.

| Hyperparameter | Meaning |
|---|---|
| `learning_rate=2e-5` | Common small learning rate for Transformer fine-tuning |
| `batch_size=8` | Small enough for a demo |
| `num_train_epochs=1` | Fast classroom demo; students may increase |
| `weight_decay=0.01` | Regularization |


In [10]:
import transformers
import accelerate

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

transformers: 5.12.1
accelerate: 1.14.0


In [ ]:
#Tells HF how to train the model, 
training_args = TrainingArguments(
    output_dir="./imdb_training_checkpoints", #where to save checkpoints
    #evaluation_strategy="epoch", #If you are using an older version of Transformers
    eval_strategy ="epoch",#evaluate the model at the end of each epoch
    save_strategy="epoch", #save the model at the end of each epoch
    learning_rate=2e-5, #the default learning rate for fine-tuning is usually between 2e-5 and 5e-5
                        #the learning rate is usually small because the model is already pretrained
    per_device_train_batch_size=8,#each device processes 8 examples at a time in training.
    per_device_eval_batch_size=8, #each device processes 8 examples at a time in testing/validation
    num_train_epochs=10,#number of epochs
    weight_decay=0.01, #Weight decay helps reduce overfitting by penalizing overly large parameter values.
    logging_steps=50,#Print or record training logs every 50 training steps.
    load_best_model_at_end=True,#After training finishes, reload the checkpoint with the best evaluation performance.
    report_to="none",#do not send training logs to any external service (like TensorBoard or Weights & Biases
)

#what model/data/metrics to use
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    #tokenizer=tokenizer, # old versions
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

### Start Training

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Evaluate the Fine-tuned Model

In [ ]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch,Accuracy
0.000003,0.266056,10,0.902680


{'eval_loss': 0.2660558223724365, 'eval_accuracy': 0.90268}


## 6. Save and reload the fine-tuned model through `pipeline()`

This is important because the assignment asks students to use the trained model with the Hugging Face `pipeline()` function.

In [ ]:
OUTPUT_DIR = "./saved_imdb_sentiment_model"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
)

print("========= Fine-tuned IMDb sentiment pipeline =========")
test_results = fine_tuned_classifier(test_texts)

for text, result in zip(test_texts, test_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== Fine-tuned IMDb sentiment pipeline ===
Text: The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.
Result: {'label': 'POSITIVE', 'score': 0.982986330986023}
--------------------------------------------------------------------------------
Text: The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.
Result: {'label': 'NEGATIVE', 'score': 0.9910921454429626}
--------------------------------------------------------------------------------
Text: What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.
Result: {'label': 'NEGATIVE', 'score': 0.977158784866333}
--------------------------------------------------------------------------------
Text: It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.
Result: {'label': 'P